In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# !pip install protobuf==3.20.3 datasets==3.0.2  --force-reinstall
# !pip uninstall pyarrow -y
# !pip install pyarrow==12.0.1
# !pip install --upgrade numpy==1.26.4 pandas==2.1.0
# !pip install --upgrade transformers

In [3]:
import json
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# --- Configuration ---
MODEL_ID = "Qwen/Qwen3-0.6B"
DATASET_ID = "gsm8k"
NUM_SAMPLES = 300
SAMPLES_PER_QUESTION = 3
OUTPUT_FILE = "/kaggle/working/gsm8k_generated_answers.jsonl"

def format_prompt(question: str) -> str:
    return f"Question: {question}\n\nLet's solve this step by step.\n\nAnswer:"

# Load model
print(f"Loading model: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load dataset
print("Loading dataset...")
dataset = load_dataset(DATASET_ID, 'main', split='train')
dataset = dataset.shuffle(seed=42).select(range(NUM_SAMPLES))

# Generate answers
print("Generating answers...")
generated_data = []

for item in tqdm(dataset):
    prompt_text = item['question']
    formatted_prompt = format_prompt(prompt_text)
    
    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        padding=False,
        truncation=False
    ).to(model.device)
    
    candidates = []
    
    for _ in range(SAMPLES_PER_QUESTION):
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=True,
                temperature=0.8,
                top_p=0.9,
                pad_token_id=tokenizer.pad_token_id,
            )
        
        response = tokenizer.decode(
            generated_ids[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        ).strip()
        
        candidates.append(response)
    
    generated_data.append({
        "question": prompt_text,
        "candidates": candidates
    })

# Save to JSONL
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for entry in generated_data:
        f.write(json.dumps(entry) + "\n")

print(f"Done! Saved {len(generated_data)} questions with generated answers to {OUTPUT_FILE}")


Loading model: Qwen/Qwen3-0.6B


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

... (13 outputs left out)

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Generating answers...


100%|██████████| 300/300 [1:59:46<00:00, 23.96s/it]

Done! Saved 300 questions with generated answers to /kaggle/working/gsm8k_generated_answers.jsonl


In [4]:
!ls /kaggle/working/

gsm8k_generated_answers.jsonl  __notebook__.ipynb


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
